# DataWorks 2026: Step-by-Step Run

Open this notebook **from the code submission folder** (the folder that contains `pipeline.py` and `Inputs_From_Phase4.zip`). In Colab, upload that whole folder as a zip if needed. Pre-computed tables are in `verified_run/`; this notebook writes a **new** run folder and does not overwrite it.

Dates and future operations remain assumptions. Conditional scenarios are not validated 20-year forecasts.

## Step 0: Install dependencies
Use a CPU runtime; no GPU is required. Training runs in subprocesses. If older NumPy/scikit-learn versions were already imported in this kernel, restart the runtime after installation and continue from Step 1.

After installing dependencies, restart the Colab session before importing packages. Then continue with Step 1 without rerunning installation. The bundled models use scikit-learn 1.6.1. A new run is separate from the preserved submitted results.


In [ ]:
%pip install -q numpy==2.1.3 pandas==2.2.3 scipy==1.16.3 scikit-learn==1.6.1 matplotlib==3.10.8 joblib==1.6.0


## Step 1: Locate the code folder and set up a new run
If this notebook is already inside the submission folder, the next cell uses it directly. In Colab, if `pipeline.py` is not found, upload **this submission folder as a zip** (must contain `pipeline.py` and `Inputs_From_Phase4.zip`).

In [ ]:
from pathlib import Path
import zipfile, io, subprocess, sys, datetime, json

def find_code_folder():
    start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        if (p / 'pipeline.py').exists() and (p / 'Inputs_From_Phase4.zip').exists():
            return p
    for p in Path.cwd().resolve().rglob('pipeline.py'):
        if (p.parent / 'Inputs_From_Phase4.zip').exists():
            return p.parent
    return None

BASE = find_code_folder()
if BASE is None:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload one zip of the submission folder (must contain pipeline.py).')
    blob = next(iter(uploaded.values()))
    extract_root = Path('/content/dataworks_' + datetime.datetime.now().strftime('%Y%m%d_%H%M%S'))
    extract_root.mkdir()
    with zipfile.ZipFile(io.BytesIO(blob)) as z:
        for name in z.namelist():
            target = (extract_root / name).resolve()
            if not target.is_relative_to(extract_root.resolve()):
                raise ValueError('Unsafe archive path')
        z.extractall(extract_root)
    BASE = next(extract_root.rglob('pipeline.py')).parent

RUN = BASE / ('run_' + datetime.datetime.now().strftime('%Y%m%d_%H%M%S'))

def run_stage(stage, *extra):
    subprocess.run([sys.executable, str(BASE/'pipeline.py'), stage, '--run', str(RUN), *extra], check=True)

print('Code folder:', BASE, '\nNew results:', RUN)
run_stage('setup', '--source', str(BASE/'Inputs_From_Phase4.zip'))

## Step 2: Audit and baseline
Review `audit_status.csv` and the figures. Keep both timing hypotheses until the date convention is confirmed. Changing a flag from False to True without evidence does not resolve an assumption.

In [ ]:
run_stage('audit')
print((RUN/'01_Audit/audit_status.csv').read_text())
from IPython.display import display, Image
display(Image(filename=str(RUN/'01_Audit/diagnostics.png')))
display(Image(filename=str(RUN/'01_Audit/baseline_match.png')))

## Step 3: Retrain the proxy
Keep the original models as references. This revision compares ExtraTrees, RandomForest, Ridge and TimeOnly; it does not claim to reproduce the original 36-model search. Cases 86-100 are not scored at this step.

In [ ]:
run_stage('train')
print((RUN/'03_Proxy/validation_summary.csv').read_text())

## Step 4: Fifteen pseudo-fields and two cutoffs
This is a computationally intensive stage. Future target-case production is hidden from history matching. Learning uses permitted data from Cases 1-70.

Inspect each fluid, not only the combined score. All 15 validation cases remain in the evaluation, including cases outside TRAIN bounds. Wait for the stage to finish.

In [ ]:
run_stage('validate')
print((RUN/'04_Validation/method_summary.csv').read_text())
display(Image(filename=str(RUN/'04_Validation/method_comparison.png')))
display(Image(filename=str(RUN/'04_Validation/backtest_Case_71.png')))

## Step 5: Download a checkpoint
Run this cell after expensive stages. Unlike the analysis stages, this download cell can be repeated.

In [ ]:
import shutil
checkpoint = shutil.make_archive(str(RUN)+'_checkpoint', 'zip', root_dir=RUN)
try:
    from google.colab import files
    files.download(checkpoint)
except ImportError:
    print('Checkpoint zip (local):', checkpoint)

## Step 6: Lock selection, then evaluate the test cases
Running this cell accepts the pre-specified validation selection rule. Make any development changes in a new run before inspecting test results. Do not tune after inspecting test and still call that evaluation untouched. Prior test inspection in the original project is disclosed.

In [ ]:
run_stage('lock')
print((RUN/'selection_lock.json').read_text())
run_stage('test')
print((RUN/'05_Test/method_summary.csv').read_text())

## Step 7: Field history matching and conditional 20-year scenarios
Evaluate original-date and +3-month query hypotheses separately. Field backtests are development evidence and do not reselect the method. Inspect recent mismatch and check for repeated forecast trajectories.

In [ ]:
run_stage('field')
print((RUN/'06_Field/status.json').read_text())
print((RUN/'06_Field/field_development_backtest.csv').read_text())
display(Image(filename=str(RUN/'06_Field/time_hypotheses.png')))
display(Image(filename=str(RUN/'06_Field/twenty_year_scenarios.png')))
display(Image(filename=str(RUN/'06_Field/cumulative_watercut.png')))

## Step 8: Export separate ten-case drafts
These are alternative drafts, not two automatically approved submissions. Resolve the date interpretation before choosing the final version. Read warnings about duplicate trajectories and submission readiness.

In [ ]:
run_stage('export', '--shift', '0')
run_stage('export', '--shift', '3')
for shift in [0, 3]:
    print((RUN/f'07_Export_Q{shift}/export_status.json').read_text())

## Step 9: Review, test and download
Generate the review tables and (in Colab) download your results. CSVs are not a PBIX; a run report is not the final PowerPoint. The video and repository upload remain separate tasks. Read `verified_run/08_Review/RUN_REVIEW.md` for the pre-computed scores.

In [ ]:
import shutil
subprocess.run([sys.executable, str(BASE/'summarize_results.py'), '--run', str(RUN)], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', str(BASE), '-p', 'test_pipeline.py', '-v'], check=True)
final_zip = shutil.make_archive(str(RUN)+'_results', 'zip', root_dir=RUN)
try:
    from google.colab import files
    files.download(final_zip)
except ImportError:
    print('Results zip (local):', final_zip)
review = RUN/'08_Review'/'RUN_REVIEW.md'
if review.exists():
    print(review.read_text())